# PII Detection for Insurance Claim Anonymization

This notebook demonstrates how to use the GLiNER PII rail in NeMo Guardrails to anonymize personally identifiable information (PII) in insurance claim narratives — keeping the fraud signal in the narrative while stripping the identity.

**Scenario.** Auto-insurance claim adjusters route claim narratives through a fraud-detection LLM. PII (claimant name, SSN, vehicle VIN, address, phone) must be masked before the narrative reaches the fraud-scoring model — both to keep the claim's signal intact and to satisfy privacy requirements.

**Three failure modes this notebook surfaces:**
- SSN missed by the rail (especially format variants without dashes)
- Over-aggressive masking that destroys the narrative ("the car was hit at [LOCATION] by [PERSON] who [ACTION]" — useless to a fraud model)
- Format-variant entities missed (SSN with dashes vs. without; phone with parens vs. dots)

The notebook walks through:
1. Configuring the GLiNER PII masking rail (local or remote deployment)
2. A smoke test on two narratives — one PII-free, one PII-rich
3. Evaluation against ~20 hand-curated insurance claim examples
4. Failure analysis surfacing the hardest false positives and false negatives

For a reference on how the GLiNER PII rail works at the configuration level (input vs. output flows, detection vs. masking modes), see the [GLiNER Integration doc](../../docs/configure-rails/guardrail-catalog/community/gliner.md).


## Local Deployment

Pull and run both NIM containers. You need an **NGC Personal API key** to pull these images and let each NIM download its model artifacts — generate one at [org.ngc.nvidia.com/setup/api-keys](https://org.ngc.nvidia.com/setup/api-keys) with at least the **NGC Catalog** service selected. Export it as `NGC_API_KEY`:

```bash
export NGC_API_KEY="<your-ngc-key>"
```

This is a different key from the `NVIDIA_API_KEY` you set earlier — they authenticate against different services (NGC for image pulls and model downloads vs. `integrate.api.nvidia.com` for hosted inference) even though both are issued by NVIDIA.

On a multi-GPU host, pin each container to a distinct GPU with `--gpus '"device=N"'` instead of `--gpus all`. Without an explicit device, both NIMs default to GPU 0 and compete for memory. The examples below assign GLiNER to GPU 0 and Llama to GPU 1; adjust the indices to match your host.

**GLiNER-PII NIM** (port 8000, GPU 0):

```bash
# Authenticate with NGC (username: $oauthtoken, password: your NGC API key)
echo $NGC_API_KEY | docker login -u '$oauthtoken' --password-stdin nvcr.io

docker run --rm -it --gpus '"device=0"' \
  -e NGC_API_KEY \
  -p 8000:8000 \
  nvcr.io/nim/nvidia/gliner-pii:1.0.0-rc1
```

**Llama 3.1 8B Instruct NIM** (port 8001, GPU 1):

```bash
docker run --rm -it --gpus '"device=1"' \
  -e NGC_API_KEY \
  -p 8001:8000 \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

Wait until both containers log `Application startup complete`, then set `DEPLOYMENT = 'local'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Remote Deployment

Set your NVIDIA API key before running the config cells:

```bash
export NVIDIA_API_KEY="nvapi-..."
```

You can obtain an API key at [build.nvidia.com](https://build.nvidia.com).

`nvidia/gliner-pii` does not appear in the configs below because it is the default value of `rails.config.gliner.model`. You only need to set that field explicitly if you want to use a different model:

```yaml
rails:
  config:
    gliner:
      model: nvidia/gliner-pii  # default — omit or change as needed
```

Alternatively, you can add  

```python
    config.rails.config.gliner.model = 'nvidia/gliner-pii' # default — omit or change as needed
```

under the `elif DEPLOYMENT == 'remote'` statements in the config cells. 

Set `DEPLOYMENT = 'remote'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Choose Deployment Type

Set `DEPLOYMENT` to `'local'` if you completed the **Local Deployment** setup above, or `'remote'` if you are using the NVIDIA-hosted endpoint.

In [1]:
# Choose local or remote deployment
DEPLOYMENT = "remote"
assert DEPLOYMENT == "local" or DEPLOYMENT == "remote", "DEPLOYMENT parameter must be set to 'local' or 'remote'"

## Import the Necessary Modules

In [2]:
import os

from nemoguardrails import LLMRails, RailsConfig

## Configure Guardrails

We use the GLiNER PII **masking on input** flow — the rail intercepts the user's claim narrative, replaces detected PII tokens with `[LABEL]` placeholders, and only the masked version reaches the LLM. The fraud-detection prompt continues to operate against an anonymized narrative.

**Why masking rather than detection?** GLiNER also offers a *detection* flow that refuses any prompt containing PII outright. For insurance claims that's the wrong default: narratives *always* contain claimant identifiers, so detection would block every legitimate request. Masking preserves the narrative signal — the description of what happened, which is what a fraud model cares about — while stripping the identity.

**Configured entity types.** The `entities` list under `gliner.input` enumerates the 12 PII categories the rail asks GLiNER to detect: names (first, last), SSN, date of birth, contact info (email, phone), address fields (street, city, state, postcode), and vehicle identifiers (VIN, license plate). Worth noting up front: the list is a *focus hint*, not a strict output filter — GLiNER may return additional types (e.g., `occupation`, `company_name`, `date`), and the evaluation section below will quantify how much that affects precision in practice.

The cell below builds a `RailsConfig` whose `server_endpoint` and `api_key_env_var` are resolved from the `DEPLOYMENT` toggle set earlier.


In [3]:
# Resolve deployment-specific endpoint and API key
if DEPLOYMENT == "remote":
    server_endpoint = "https://integrate.api.nvidia.com/v1/chat/completions"
    api_key_env_var = "NVIDIA_API_KEY"
    llama_base_url = None
elif DEPLOYMENT == "local":
    server_endpoint = "http://localhost:8000/v1/chat/completions"
    api_key_env_var = None
    llama_base_url = "http://localhost:8001/v1"

# Build the YAML conditionally; insurance-claim PII masking flow with the GLiNER NIM.
config_yaml = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct
"""
if DEPLOYMENT == "remote":
    config_yaml += "    api_key_env_var: NVIDIA_API_KEY\n"
elif DEPLOYMENT == "local":
    config_yaml += f"    parameters:\n      base_url: {llama_base_url}\n"

config_yaml += f"""
rails:
  config:
    gliner:
      server_endpoint: {server_endpoint}
"""
if api_key_env_var:
    config_yaml += f"      api_key_env_var: {api_key_env_var}\n"

config_yaml += """      threshold: 0.5
      input:
        entities:
          - first_name
          - last_name
          - ssn
          - date_of_birth
          - email
          - phone_number
          - street_address
          - city
          - state
          - postcode
          - vehicle_identifier
          - license_plate
  input:
    flows:
      - gliner mask pii on input
"""

config = RailsConfig.from_content(yaml_content=config_yaml)
rails = LLMRails(config)
print(f"Rail wired up against the {DEPLOYMENT} GLiNER endpoint.")

Rail wired up against the remote GLiNER endpoint.


## Smoke Test

The cell below sends two narratives through the rail:

1. **PII-free (partially masked due to over-broad detection)** — a hail-damage report that mentions a vehicle make/model but no claimant identifiers. Ideally this would pass through to the LLM unchanged, but the GLiNER NIM's over-broad `vehicle_identifier` detection tags "2018 Subaru Outback" as a vehicle identifier and masks it to `[VEHICLE_IDENTIFIER]`. This is one of the over-masking patterns the eval section quantifies — not a setup problem.
2. **PII-rich** — a rear-end-collision narrative with a claimant name, VIN, email, and phone number. Each should be replaced with a `[LABEL]` placeholder before the LLM sees it.

**Read the rail trace, not the LLM response.** The rail execution trace printed after each call shows the *masked* version of the prompt that the LLM actually received. For the PII-rich case, you should see `[FIRST_NAME] [LAST_NAME] ... [VEHICLE_IDENTIFIER]` in place of the original tokens. For the PII-free case, the trace reveals the `[VEHICLE_IDENTIFIER]` replacement that the LLM response alone doesn't make obvious.

The LLM's response itself often invents plausible-looking PII to fill the placeholders — it sees `[FIRST_NAME]` as literal text, not as a redaction marker, and tries to be helpful by generating a name. **This is expected behavior** for a vanilla chat LLM and is harmless here because the *original* PII never reached the LLM. Production deployments downstream of the rail need to handle this explicitly: either prompt the LLM to treat `[LABEL]` tokens as redactions, or post-process the response to strip any generated identifiers.

**What to check if it's not working.** If the rail trace doesn't show `gliner mask pii on input` firing, the flow isn't wired up — verify GLiNER endpoint reachability, API key propagation into the Jupyter kernel (`import os; print(os.environ.get('NVIDIA_API_KEY'))` from a notebook cell), and that the `input.flows` list in the config cell above is spelled exactly as `gliner mask pii on input`.


In [4]:
# Smoke-test the rail with two insurance-claim narratives:
#   1. PII-free  -> ideally passes through unchanged (but vehicle_identifier gets over-masked)
#   2. PII-rich  -> should be masked before the LLM sees it

clean_narrative = (
    "Hail damage report: 2018 Subaru Outback with multiple dents across the "
    "hood and roof. Estimated repair cost $2,400. No injuries."
)

pii_rich_narrative = (
    "Claimant Mary Johnson reported a rear-end collision involving her 2019 "
    "Toyota Camry (VIN 1HGCM82633A123456). Contact: mary.johnson@example.com "
    "or (415) 555-0123."
)

for label, prompt in [("PII-free", clean_narrative), ("PII-rich", pii_rich_narrative)]:
    print(f"=== {label} narrative ===")
    print(f"User prompt:\n  {prompt}\n")
    response = await rails.generate_async(messages=[{"role": "user", "content": prompt}])
    print(f"LLM response (against the rail-processed prompt):\n  {response['content']}\n")
    info = rails.explain()
    print("Rail execution trace:")
    print(info.colang_history)
    print()

=== PII-free narrative ===
User prompt:
  Hail damage report: 2018 Subaru Outback with multiple dents across the hood and roof. Estimated repair cost $2,400. No injuries.

LLM response (against the rail-processed prompt):
  It sounds like there's been a significant impact on the vehicle, and I'm here to help you process the hail damage report. 

Let's break down the details you've provided: the vehicle, which we'll refer to as [VEHICLE_IDENTIFIER], has sustained multiple dents across both the hood and the roof. This type of damage can be quite extensive and may require a thorough inspection to determine the full extent of the repairs needed.

The estimated repair cost comes out to be $2,400. This is a substantial amount, and it's essential to consider whether the cost of repairs will be covered by insurance or if you'll need to absorb the expense yourself.

Fortunately, it's great to hear that there were no injuries as a result of the hail damage. This is always the top priority, and I

## Evaluation against an Insurance Claim Subset

The rest of the notebook evaluates the rail against a hand-curated set of ~20 insurance-claim narratives bundled with this notebook at `data/pii_detection_subset.csv`. Each row carries both the input narrative and the ground truth entity spans, but the rail runs against the input column only — ground truth is referenced for the first time when we compute metrics.

Flip `USE_FULL_DATASET = True` in the cell below to run against the full AI4Privacy PII-Masking-300k dataset from HuggingFace; the in-repo subset is the default for a quick sanity check.


In [5]:
import json

import pandas as pd

USE_FULL_DATASET = False  # set True to download the full AI4Privacy PII-Masking-300k from HF

if USE_FULL_DATASET:
    from datasets import load_dataset

    ds = load_dataset("ai4privacy/pii-masking-300k", split="train")
    df = ds.to_pandas()
    # Filter to insurance-adjacent rows. Column names vary across AI4Privacy snapshots —
    # confirm with ds.column_names before relying on a specific field.
    if "domain" in df.columns:
        df = df[df["domain"].str.contains("insurance|claim|financial", case=False, na=False)]
    df = df.reset_index(drop=True)
else:
    df = pd.read_csv("data/pii_detection_subset.csv")
    # Parse the entities column from JSON string to Python list. This is the ground truth
    # column — the next cell ("Run the rail") will not read it. Metrics will.
    df["entities"] = df["entities"].apply(json.loads)

print(f"Loaded {len(df)} examples")
df[["example_id", "text"]].head()

Loaded 20 examples


,example_id,text
0,pii_001,Claimant Mary Johnson reported a rear-end coll...
1,pii_002,Policy holder William O'Brien filed for theft ...
2,pii_003,"Sarah Chen, DOB 04/12/1985, reports vandalism ..."
3,pii_004,Insured: Marcus Williams. Primary phone (208) ...
4,pii_005,Hit-and-run incident reported by Daniel Park (...


### Running the Rail

The cell below sends each narrative through GLiNER and captures the entity predictions. Two implementation notes worth knowing:

- **We call `gliner_request` directly** rather than going through `rails.generate(...)`. The masking rail invokes the same function internally, but only exposes the post-masking text; for span-level evaluation we need the raw entities (with `start_position`, `end_position`, `suggested_label`) that the function returns before the masking step.
- **The hosted endpoint is rate-limited.** For `DEPLOYMENT = 'remote'` the loop throttles to ~4 requests per second with a small inter-call sleep, and retries any HTTP 429 with exponential backoff (sleeps 1, 2, 4, 8, 16 s between `MAX_RETRIES = 6` attempts; the final attempt raises rather than sleeping). For `DEPLOYMENT = 'local'` the throttle drops to 0 — the in-process NIM has no rate limit, so the sleep would be pure overhead.


In [6]:
import asyncio

from tqdm.auto import tqdm

from nemoguardrails.library.gliner.request import gliner_request

# Throttled loop with exponential backoff on 429 rate-limit responses.
# build.nvidia.com's hosted endpoint caps sustained throughput; a local NIM deployment
# doesn't have this constraint, so this throttle is only meaningful for DEPLOYMENT='remote'.

MAX_RETRIES = 6
THROTTLE_S = 0.25 if DEPLOYMENT == "remote" else 0.0  # ~4 req/sec on remote; no throttle on local


async def detect_with_retry(text):
    """Call gliner_request, retrying with exponential backoff on 429 errors.

    Catches a broad Exception (not just ValueError) so the retry behavior stays consistent
    if gliner_request's 429 path changes exception class in a future nemoguardrails release —
    and to match the sibling notebooks (content_safety_nim.ipynb, topic_control_nim.ipynb).
    The `"429" in str(e)` check guards against retrying non-429 errors.

    Sleeps between attempts are 2**attempt seconds: (1, 2, 4, 8, 16). The final attempt
    (attempt == MAX_RETRIES - 1) raises rather than sleeping, so the sleep sequence has
    MAX_RETRIES - 1 = 5 elements across MAX_RETRIES = 6 total attempts. The function
    either returns from the try branch or re-raises from the except branch on the last
    attempt — there is no post-loop return path.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = await gliner_request(
                text=text,
                server_endpoint=server_endpoint,
                api_key=os.getenv(api_key_env_var) if api_key_env_var else None,
                threshold=0.5,
                model="nvidia/gliner-pii",
            )
            return response.get("entities", [])
        except Exception as e:
            if "429" not in str(e) or attempt == MAX_RETRIES - 1:
                raise
            await asyncio.sleep(2**attempt)  # 1, 2, 4, 8, 16 s


predicted_entities_per_row = []
for text in tqdm(df["text"], desc="GLiNER inference"):
    try:
        entities = await detect_with_retry(text)
        predicted_entities_per_row.append(entities)
    except Exception as e:
        print(f"  Error on text {text[:60]!r}: {e}")
        predicted_entities_per_row.append(None)
    await asyncio.sleep(THROTTLE_S)

df["predicted_entities"] = predicted_entities_per_row
n_classified = sum(1 for e in predicted_entities_per_row if e is not None)
print(f"GLiNER produced entity predictions on {n_classified}/{len(df)} rows")

/Users/schilton/venv-guardrails-docs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GLiNER inference: 100%|█████████████████████████████████████████████| 20/20 [00:17<00:00,  1.15it/s]

GLiNER produced entity predictions on 20/20 rows


### Computing Metrics

Now compare predictions against ground truth. The matching policy is **type-equal + any character-level span overlap counts as a true positive** — lenient enough to handle minor boundary disagreements (e.g., GLiNER tagging `Mary Johnson` as one span while ground truth tags `Mary` and `Johnson` separately).

For each gold entity, the cell finds the first unmatched predicted entity of the same type whose span overlaps. If found, that's a TP and the prediction is consumed. Predictions that never get matched to a gold are FPs; gold entities that never find a matching prediction are FNs.

The cell reports:

- **Entity-level precision / recall / F1** — the headline numbers
- **Over-masking rate** = fraction of predicted entities not in ground truth (= `1 - precision`; surfaced separately because "over-masking" is the natural framing for the insurance-anonymization scenario, where extra masks destroy narrative signal)
- **Per-entity-type recall** broken out so you can see which entity types the rail handles reliably and which it misses


In [7]:
from collections import defaultdict

# Exclude rows where the GLiNER call failed (predicted_entities is None) so errored
# rows don't inflate FN counts. Matches the sibling notebooks' dropna pattern.
valid = df.dropna(subset=["predicted_entities"])


def span_match(gold_entity, predicted_entity):
    """A predicted entity matches a gold entity if the type is equal and the spans overlap."""
    if gold_entity["type"] != predicted_entity.get("suggested_label", ""):
        return False
    gold_s, gold_e = gold_entity["start"], gold_entity["end"]
    pred_s = predicted_entity.get("start_position", 0)
    pred_e = predicted_entity.get("end_position", 0)
    return gold_s < pred_e and pred_s < gold_e  # any character-level overlap counts


tp, fp, fn = 0, 0, 0
per_type_tp = defaultdict(int)
per_type_fn = defaultdict(int)

for _, row in valid.iterrows():
    gold = row["entities"]
    predicted = row["predicted_entities"]
    matched_pred_idx = set()
    for g in gold:
        hit = next(
            (i for i, p in enumerate(predicted) if i not in matched_pred_idx and span_match(g, p)),
            None,
        )
        if hit is not None:
            tp += 1
            per_type_tp[g["type"]] += 1
            matched_pred_idx.add(hit)
        else:
            fn += 1
            per_type_fn[g["type"]] += 1
    fp += len(predicted) - len(matched_pred_idx)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
over_masking_rate = fp / (tp + fp) if (tp + fp) else 0.0  # = 1 - precision

print("=" * 60)
print(f"Entity-level   precision: {precision:.3f}   recall: {recall:.3f}   F1: {f1:.3f}")
print(f"               TP: {tp}   FP: {fp}   FN: {fn}")
print(f"               Over-masking rate: {over_masking_rate:.3f}")
print("=" * 60)
print()
print("Per-entity-type recall:")
all_types = sorted(set(list(per_type_tp.keys()) + list(per_type_fn.keys())))
for t in all_types:
    t_tp = per_type_tp[t]
    t_fn = per_type_fn[t]
    t_rec = t_tp / (t_tp + t_fn) if (t_tp + t_fn) else 0.0
    bar = "█" * int(t_rec * 20)
    print(f"  {t:<22} {t_tp}/{t_tp + t_fn}  ({t_rec:.2f})  {bar}")

Entity-level   precision: 0.731   recall: 0.990   F1: 0.841
               TP: 98   FP: 36   FN: 1
               Over-masking rate: 0.269

Per-entity-type recall:
  city                   9/9  (1.00)  ████████████████████
  date_of_birth          2/2  (1.00)  ████████████████████
  email                  9/9  (1.00)  ████████████████████
  first_name             18/18  (1.00)  ████████████████████
  last_name              18/18  (1.00)  ████████████████████
  license_plate          1/1  (1.00)  ████████████████████
  phone_number           11/12  (0.92)  ██████████████████
  postcode               8/8  (1.00)  ████████████████████
  ssn                    3/3  (1.00)  ████████████████████
  state                  9/9  (1.00)  ████████████████████
  street_address         7/7  (1.00)  ████████████████████
  vehicle_identifier     3/3  (1.00)  ████████████████████


### Failure Analysis

The headline F1 tells you whether the rail is good on aggregate; it doesn't say *which* errors matter. The next cell sorts the rows by false-negative count and false-positive count and surfaces the top 3 of each — where the rail's mistakes are concentrated.

For false negatives, this shows real PII spans that slipped through. For false positives, it shows what the rail flagged that ground truth says isn't PII (often the most instructive output — over-masking patterns tend to cluster by entity type and confidence). The output drives the discussion that follows.


In [8]:
# Surface the hardest false negatives (missed PII) and false positives (over-masking).
# This is where a reader sees which kinds of entities the rail struggles with.
# Uses `valid` from the metrics cell above (rows where GLiNER returned predictions).

failures = []
for _, row in valid.iterrows():
    gold = row["entities"]
    predicted = row["predicted_entities"]
    matched_pred_idx = set()
    fns_local = []
    for g in gold:
        hit = next(
            (i for i, p in enumerate(predicted) if i not in matched_pred_idx and span_match(g, p)),
            None,
        )
        if hit is not None:
            matched_pred_idx.add(hit)
        else:
            fns_local.append(g)
    fps_local = [predicted[i] for i in range(len(predicted)) if i not in matched_pred_idx]
    failures.append({"example_id": row["example_id"], "text": row["text"], "fns": fns_local, "fps": fps_local})

print("=== Top 3 rows by missed PII (false negatives) ===")
for r in sorted(failures, key=lambda x: -len(x["fns"]))[:3]:
    if not r["fns"]:
        continue
    print(f"\n[{r['example_id']}] {r['text'][:140]}{'...' if len(r['text']) > 140 else ''}")
    for fn_ in r["fns"]:
        print(f"  MISSED: {fn_['type']:<22} '{fn_['text']}'   chars [{fn_['start']}:{fn_['end']}]")

print("\n=== Top 3 rows by over-masking (false positives) ===")
for r in sorted(failures, key=lambda x: -len(x["fps"]))[:3]:
    if not r["fps"]:
        continue
    print(f"\n[{r['example_id']}] {r['text'][:140]}{'...' if len(r['text']) > 140 else ''}")
    for fp_ in r["fps"]:
        label = fp_.get("suggested_label", "?")
        value = fp_.get("value", "?")
        score = fp_.get("score", 0)
        print(f"  EXTRA:  {label:<22} '{value}'   (score {score:.2f})")

=== Top 3 rows by missed PII (false negatives) ===

[pii_004] Insured: Marcus Williams. Primary phone (208) 555-0125. Cell 208-555-0126. Email m.williams@example.org. Mailing address: 88 Cedar Lane, Boi...
  MISSED: phone_number           '208-555-0126'   chars [61:73]

=== Top 3 rows by over-masking (false positives) ===

[pii_017] Contact policyholder David Kim at david.kim+claims@example.com or work line (206) 555-0131 ext 213. Personal cell: 206-555-0132.
  EXTRA:  occupation             'policyholder'   (score 0.77)
  EXTRA:  first_name             'david'   (score 1.00)
  EXTRA:  user_name              'david.kim'   (score 0.90)
  EXTRA:  last_name              'kim'   (score 1.00)
  EXTRA:  phone_number           '(206) 555-0131 ext 213'   (score 0.81)

[pii_018] Commercial fleet claim from Henderson Logistics Inc. Fleet manager: Thomas Reeves, t.reeves@example-logistics.com, (469) 555-0133. Vehicle: ...
  EXTRA:  company_name           'Henderson Logistics Inc'   (score 0.51)
 

## Discussion

The rail catches PII reliably — **entity-level recall is 99%** (98/99 entities found), with only one phone-number variant missed (`208-555-0126` in row 4, while the cap-cased `(208) 555-0125` in the same row was caught). All 11 of the other configured entity types hit 100% recall on the in-repo subset. For the insurance scenario, that's the headline good news: a claim narrative passed through this rail will reliably emerge with its identifying tokens redacted.

The precision side tells the more interesting story. **Precision is 0.73 and over-masking rate is 0.27** — roughly one in four of the rail's predictions is something the ground truth doesn't consider PII. Three distinct over-masking patterns drive this:

1. **Out-of-scope entity types** (the biggest source). Even though the rail's `entities` list specifies 12 types (`first_name`, `last_name`, `ssn`, …, `license_plate`), GLiNER returns predictions tagged `occupation` ("policyholder", "Fleet manager"), `user_name` ("david.kim"), `company_name` ("Henderson Logistics Inc"), and `date` ("March 14") in addition. **The configured `entities` list filters which entity types the rail asks GLiNER to focus on, but does not restrict GLiNER's output to those types only.** Downstream consumers see extra masks they never requested.
2. **Double-detection of email-local components.** In row 17, GLiNER tags the cap-cased "David" and "Kim" as `first_name`/`last_name` (correctly matching ground truth), then *also* tags the lowercase "david" and "kim" inside the address `david.kim+claims@example.com` as separate `first_name`/`last_name` entities. The email span itself is tagged correctly; the substrings inside it become bonus FPs.
3. **Over-broad notion of certain types.** `vehicle_identifier` reaches beyond VINs to grab "2023 Freightliner Cascadia" (and "2018 Subaru Outback" in the smoke test); `street_address` extends to road-intersection descriptions like "Oak Street and Elm Avenue" that lack a house number; the `phone_number` span swallows the extension `ext 213`.

**For the insurance fraud-detection scenario specifically**, the precision hit matters: when a fraud-scoring model receives `[COMPANY_NAME] [OCCUPATION] reports [VEHICLE_IDENTIFIER] sustained damage at [STREET_ADDRESS] on [DATE]`, the narrative signal that drives fraud detection (company / role / vehicle type / incident location / timing) has been destroyed. High recall is great for privacy; low precision is bad for downstream utility. Production deployments need to balance the two deliberately.

## Next steps

- **Post-filter to the configured entity list.** Wrap the rail's output in a downstream filter that drops any predicted entity whose type isn't in the configured `entities` list. Closes the #1 over-masking source in one shot. (Open question: whether to land this at the rail level or the masking-action level — the latter is cleaner.)
- **Tune `threshold`.** Currently 0.5. Raising to 0.7 would suppress the lower-confidence FPs (`policyholder` @ 0.77, `company_name` @ 0.51). Trade-off: some real PII drops too.
- **Investigate `flat_ner=True`.** Currently false. Toggling may suppress the double-detection of email-local substrings by forcing GLiNER to pick one span per character position.
- **Run against the full AI4Privacy dataset** (`USE_FULL_DATASET = True`). Confirm whether these patterns hold at scale — the in-repo subset is 20 rows; a 300k-row run will surface tail behaviors. Note: rate limiting on the hosted endpoint means the local-deployment path (`DEPLOYMENT='local'`) is the right way to do this at scale.
- **Custom entity-type filter at the GLiNER request level.** Some GLiNER variants accept a strict-mode parameter that restricts the model's output to a passed-in label list. Worth checking whether `nvidia/gliner-pii` supports this — if so, it would obviate the post-filter approach above.
- **Pair the rail with rule-based validators** for high-stakes types: e.g., reject `vehicle_identifier` predictions that don't match a 17-character alphanumeric VIN pattern; reject `phone_number` extensions; require `street_address` to start with a house number. Pragmatic but reduces the rail's elegance.

The trade-off direction depends entirely on the downstream consumer: if it's a privacy-compliance model that cares only about catching all PII, the current rail is in good shape. If it's a fraud-detection model that needs narrative context, the precision deficit is the headline problem to address.


## Supported Entity Types

The NVIDIA GLiNER-PII NIM supports these PII categories:

| Category | Entity Types |
|----------|-------------|
| Personal Identifiers | `first_name`, `last_name`, `ssn`, `date_of_birth`, `age`, `gender` |
| Contact Information | `email`, `phone_number`, `fax_number`, `street_address`, `city`, `state`, `postcode`, `country`, `county` |
| Financial | `credit_debit_card`, `cvv`, `bank_routing_number`, `account_number`, `swift_bic`, `tax_id` |
| Technical | `ipv4`, `ipv6`, `mac_address`, `url`, `api_key`, `password`, `pin`, `http_cookie` |
| Identification | `national_id`, `license_plate`, `vehicle_identifier`, `employee_id`, `customer_id`, `unique_id` |
| Sensitive Attributes | `sexuality`, `political_view`, `race_ethnicity`, `religious_belief`, `blood_type` |
